In [2]:
pip install autogen ag2[openai] --q

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 0.41.1 requires langchain<=0.3,>0.2, but you have langchain 0.3.25 which is incompatible.
crewai 0.41.1 requires openai<2.0.0,>=1.13.3, but you have openai 2.38.0 which is incompatible.
embedchain 0.1.116 requires langchain<=0.3,>0.2, but you have langchain 0.3.25 which is incompatible.
embedchain 0.1.116 requires langchain-community<0.3.0,>=0.2.6, but you have langchain-community 0.3.24 which is incompatible.
embedchain 0.1.116 requires langchain-openai<0.2.0,>=0.1.7, but you have langchain-openai 0.3.16 which is incompatible.
instructor 1.3.3 requires jiter<0.5.0,>=0.4.1, but you have jiter 0.14.0 which is incompatible.
instructor 1.3.3 requires openai<2.0.0,>=1.1.0, but you have openai 2.38.0 which is incompatible.
langchain-cohere 0.1.9 requires langchain-core<0.3,>=0.2.2, but you have langchain-core 0.

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
# may struggle for local setup highly recommended to run in vocarium
import autogen
print("AutoGen Installed Successfully:", autogen.__version__)
import utils
import os

from autogen import ConversableAgent

AutoGen Installed Successfully: 0.13.1


# <strong> Conversable Agent </strong>

In [7]:
agent = ConversableAgent(
    "chatbot",
    llm_config={"config_list": [{"model": "gpt-4o-mini", 
                                 "api_key": os.environ.get("OPENAI_API_KEY"),
                                 "base_url" : os.environ.get("OPENAI_API_BASE")
                                }]},
    code_execution_config=False,  # Turn off code execution, by default it is off.
    function_map=None,  # No registered functions, by default it is None.
    human_input_mode="NEVER",  # Never ask for human input.
)

In [8]:
reply = agent.generate_reply(messages=[{"content": "Tell me a joke.", "role": "user"}])
print(reply)

Why don't scientists trust atoms? 

Because they make up everything!


# <strong> Two agents </strong>

In [9]:
cathy = ConversableAgent(
    "cathy",
    system_message="Your name is Cathy and you are a part of a duo of comedians.",
    llm_config={"config_list": [{"model": "gpt-4o-mini", 
                                 "temperature": 0.9, 
                                 "api_key": os.environ.get("OPENAI_API_KEY"),
                                 "base_url": os.environ.get("OPENAI_API_BASE")
                                }]},
    human_input_mode="NEVER",  # Never ask for human input.
)

joe = ConversableAgent(
    "joe",
    system_message="Your name is Joe and you are a part of a duo of comedians.",
    llm_config={"config_list": [{"model": "gpt-4o-mini", 
                                 "temperature": 0.7, 
                                 "api_key": os.environ.get("OPENAI_API_KEY"),
                                 "base_url": os.environ.get("OPENAI_API_BASE")
                                }]},
    human_input_mode="NEVER",  # Never ask for human input.
)

In [10]:
result = joe.initiate_chat(cathy, message="Cathy, tell me a joke.", max_turns=2)

joe (to cathy):

Cathy, tell me a joke.

--------------------------------------------------------------------------------
cathy (to joe):

Sure, Joe! Why did the scarecrow win an award? 

Because he was outstanding in his field!

--------------------------------------------------------------------------------
joe (to cathy):

That’s a classic, Cathy! Alright, here’s one for you: Why don’t skeletons fight each other? Because they don’t have the guts!

--------------------------------------------------------------------------------
cathy (to joe):

Haha, that's a good one, Joe! I've got another for you: Why did the bicycle fall over? 

Because it was two-tired!

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (ed282c36-2f81-4be1-9b1f-e009a233ed1e): Maximum turns (2) reached


# <strong> Two Agents with Human Intervention </strong>

In [11]:
SYSTEM_PROMPT_WITH_NUMBER = """
You are playing a game of guess-my-number. In the first game, you have the number 53 in your mind, and I will try to guess it.
If I guess too high, say 'too high', if I guess too low, say 'too low'.
"""

SYSTEM_PROMPT_GUESS = """
I have a number in my mind, and you will try to guess it. If I say 'too high', you should guess a lower number. 
If I say 'too low', you should guess a higher number.
"""

agent_with_number = ConversableAgent(
    "agent_with_number",
    system_message=SYSTEM_PROMPT_WITH_NUMBER,
    llm_config={"config_list": [{"model": "gpt-4", 
                                 "api_key": os.environ["OPENAI_API_KEY"],
                                 "base_url": os.environ["OPENAI_API_BASE"]
                                }]},
    max_consecutive_auto_reply=1,  # maximum number of consecutive auto-replies before asking for human input
    is_termination_msg=lambda msg: "53" in msg["content"],  # terminate if the number is guessed by the other agent
    human_input_mode="TERMINATE",  # ask for human input until the game is terminated
)

agent_guess_number = ConversableAgent(
    "agent_guess_number",
    system_message=SYSTEM_PROMPT_GUESS,
    llm_config={"config_list": [{"model": "gpt-4", 
                                 "api_key": os.environ["OPENAI_API_KEY"],
                                 "base_url": os.environ["OPENAI_API_BASE"]
                                }]},
    human_input_mode="NEVER",
)

In [13]:
result = agent_with_number.initiate_chat(
    agent_guess_number,
    message="I have a number between 1 and 100. Guess it!",
)

agent_with_number (to agent_guess_number):

I have a number between 1 and 100. Guess it!

--------------------------------------------------------------------------------
agent_guess_number (to agent_with_number):

Is the number 50?

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
agent_with_number (to agent_guess_number):

Too low.

--------------------------------------------------------------------------------
agent_guess_number (to agent_with_number):

Is it 75?

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (62cb9eb1-6a72-4299-b958-9d0a1017ce36): User requested to end the conversation
